# 2. ETL Pipeline - Xử lý và Biến đổi Dữ liệu

Notebook này trình bày toàn bộ quy trình **ETL (Extract - Transform - Load)** cho dữ liệu Olist E-Commerce:
- **Extract**: Đọc 9 bảng dữ liệu từ HDFS Bronze
- **Transform**: Join bảng, làm sạch, tạo features mới (delivery_days, RFM, churn)
- **Load**: Lưu vào HDFS Silver/Gold dưới dạng Parquet

## 2.1 Khởi tạo PySpark Session

In [14]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/Admin/AppData/Local/Programs/Python/Python313/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["TZ"] = "UTC"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Khởi tạo SparkSession
spark = (
    SparkSession.builder
    .appName("Olist_ETL_Notebook")
    .master("local[4]")               # Dùng 4 core CPU
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("SparkSession đã khởi tạo thành công!")
print(f"   Spark version: {spark.version}")

SparkSession đã khởi tạo thành công!
   Spark version: 3.5.1


## 2.2 Đọc dữ liệu từ HDFS Bronze

Đọc 9 file CSV đã được upload ở bước Ingestion.

| Bảng | Vai trò | Mục đích |
|------|---------|----------|
| orders | Bảng chính | Chứa thông tin đơn hàng, trạng thái, thời gian |
| order_items | Chi tiết | Sản phẩm, giá, phí vận chuyển trong mỗi đơn |
| payments | Thanh toán | Phương thức, giá trị thanh toán |
| reviews | Đánh giá | Điểm đánh giá của khách hàng |
| customers | Khách hàng | Thông tin vị trí khách hàng |
| products | Sản phẩm | Thông tin sản phẩm, danh mục |
| sellers | Người bán | Thông tin vị trí người bán |
| geolocation | Địa lý | Tọa độ GPS |
| category_translation | Dịch thuật | Dịch tên danh mục PT→EN |

In [15]:
# Đường dẫn HDFS Bronze
HDFS_BRONZE = "hdfs://localhost:9000/user/bigdata/olist/bronze"

# Đọc tất cả các bảng
csv_files = {
    "orders":       "orders/olist_orders_dataset.csv",
    "order_items":  "order_items/olist_order_items_dataset.csv",
    "payments":     "order_payments/olist_order_payments_dataset.csv",
    "reviews":      "order_reviews/olist_order_reviews_dataset.csv",
    "customers":    "customers/olist_customers_dataset.csv",
    "products":     "products/olist_products_dataset.csv",
    "sellers":      "sellers/olist_sellers_dataset.csv",
    "geolocation":  "geolocation/olist_geolocation_dataset.csv",
    "category_translation": "category_translation/product_category_name_translation.csv",
}

dataframes = {}
for name, filename in csv_files.items():
    path = f"{HDFS_BRONZE}/{filename}"
    print(f" Đang đọc: {name}...")
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    dataframes[name] = df
    print(f"   -> {name}: {df.count():,} dòng, {len(df.columns)} cột")

print(f"\nĐã đọc thành công {len(dataframes)} bảng dữ liệu!")

 Đang đọc: orders...
   -> orders: 99,441 dòng, 8 cột
 Đang đọc: order_items...
   -> order_items: 112,650 dòng, 7 cột
 Đang đọc: payments...
   -> payments: 103,886 dòng, 5 cột
 Đang đọc: reviews...
   -> reviews: 104,162 dòng, 7 cột
 Đang đọc: customers...
   -> customers: 99,441 dòng, 5 cột
 Đang đọc: products...
   -> products: 32,951 dòng, 9 cột
 Đang đọc: sellers...
   -> sellers: 3,095 dòng, 4 cột
 Đang đọc: geolocation...
   -> geolocation: 1,000,163 dòng, 5 cột
 Đang đọc: category_translation...
   -> category_translation: 71 dòng, 2 cột

Đã đọc thành công 9 bảng dữ liệu!


## 2.3 Join các bảng dữ liệu

Các bảng được join theo sơ đồ sau:
```
orders (bảng chính)
  ├── customers (LEFT JOIN on customer_id)
  ├── order_items → products → category_translation
  ├── payments (GROUP BY order_id trước khi join)
  ├── reviews (GROUP BY order_id trước khi join)
  └── sellers
```

In [16]:
# === BƯỚC 2a: Tổng hợp payments theo order_id ===
print("2a. Tổng hợp payments theo order_id...")
payments_agg = (
    dataframes["payments"]
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.max("payment_installments").alias("max_installments"),
        F.count("*").alias("payment_count"),
        F.first("payment_type").alias("payment_type"),
        F.collect_set("payment_type").alias("payment_types_used"),
    )
)
print(f"   -> payments_agg: {payments_agg.count():,} dòng")
payments_agg.show(3, truncate=False)

2a. Tổng hợp payments theo order_id...
   -> payments_agg: 99,440 dòng
+--------------------------------+-------------------+----------------+-------------+------------+------------------+
|order_id                        |total_payment_value|max_installments|payment_count|payment_type|payment_types_used|
+--------------------------------+-------------------+----------------+-------------+------------+------------------+
|000229ec398224ef6ca0657da4fc703e|216.87             |5               |1            |credit_card |[credit_card]     |
|00042b26cf59d7ce69dfabb4e55b4fd9|218.04             |3               |1            |credit_card |[credit_card]     |
|0005f50442cb953dcd1d21e1fb923495|65.39              |1               |1            |credit_card |[credit_card]     |
+--------------------------------+-------------------+----------------+-------------+------------+------------------+
only showing top 3 rows



In [17]:
# === BƯỚC 2b: Tổng hợp reviews theo order_id ===
print("2b. Tổng hợp reviews theo order_id...")
reviews_agg = (
    dataframes["reviews"]
    .groupBy("order_id")
    .agg(
        F.avg("review_score").alias("review_score"),
        F.first("review_comment_title").alias("review_comment_title"),
        F.first("review_comment_message").alias("review_comment_message"),
        F.count("*").alias("review_count"),
    )
)
print(f"   -> reviews_agg: {reviews_agg.count():,} dòng")

2b. Tổng hợp reviews theo order_id...
   -> reviews_agg: 99,743 dòng


In [18]:
# === BƯỚC 2c: Dịch tên danh mục sản phẩm sang tiếng Anh ===
print("2c. Dịch tên danh mục sản phẩm...")
products_translated = dataframes["products"].join(
    dataframes["category_translation"],
    on="product_category_name",
    how="left"
)

# === BƯỚC 2d: Join order_items với products và sellers ===
print("2d. Join order_items + products + sellers...")
items_with_products = dataframes["order_items"].join(products_translated, on="product_id", how="left")
items_with_sellers = items_with_products.join(
    dataframes["sellers"].select("seller_id", "seller_city", "seller_state", "seller_zip_code_prefix"),
    on="seller_id", how="left"
)

# === BƯỚC 2e: Tổng hợp order_items theo order_id ===
print("2e. Tổng hợp order_items theo order_id...")
items_agg = (
    items_with_sellers
    .groupBy("order_id")
    .agg(
        F.count("*").alias("total_items"),
        F.sum("price").alias("total_price"),
        F.sum("freight_value").alias("total_freight_value"),
        F.countDistinct("product_id").alias("unique_products"),
        F.countDistinct("seller_id").alias("unique_sellers"),
        F.first("product_category_name_english").alias("main_category_english"),
        F.first("product_category_name").alias("main_category"),
        F.first("seller_city").alias("seller_city"),
        F.first("seller_state").alias("seller_state"),
        F.avg("product_weight_g").alias("avg_product_weight_g"),
    )
)
print(f"   -> items_agg: {items_agg.count():,} dòng")

2c. Dịch tên danh mục sản phẩm...
2d. Join order_items + products + sellers...
2e. Tổng hợp order_items theo order_id...
   -> items_agg: 98,666 dòng


In [19]:
# === BƯỚC 2f: JOIN CHÍNH - Gộp tất cả lại ===
print("2f. Thực hiện join cuối cùng...")
merged = (
    dataframes["orders"]
    .join(dataframes["customers"], on="customer_id", how="left")
    .join(items_agg, on="order_id", how="left")
    .join(payments_agg, on="order_id", how="left")
    .join(reviews_agg, on="order_id", how="left")
)

print(f"\n merged_orders: {merged.count():,} dòng, {len(merged.columns)} cột")
print(f"   Danh sách cột: {merged.columns}")
merged.show(5)

2f. Thực hiện join cuối cùng...

 merged_orders: 99,441 dòng, 31 cột
   Danh sách cột: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_items', 'total_price', 'total_freight_value', 'unique_products', 'unique_sellers', 'main_category_english', 'main_category', 'seller_city', 'seller_state', 'avg_product_weight_g', 'total_payment_value', 'max_installments', 'payment_count', 'payment_type', 'payment_types_used', 'review_score', 'review_comment_title', 'review_comment_message', 'review_count']
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+-------------+--------------+-----

## 2.4 Làm sạch dữ liệu (Data Cleaning)

### Các vấn đề cần xử lý:
1. **Bản ghi trùng lặp**: Cùng `order_id` xuất hiện nhiều lần → `dropDuplicates`
2. **Parse ngày tháng**: Chuỗi text → kiểu Timestamp để tính toán được
3. **Giá trị null**: Điền giá trị mặc định phù hợp (0 cho số, "unknown" cho text)
4. **Đơn hàng không hợp lệ**: Loại bỏ đơn bị hủy không có thông tin

### Tại sao KHÔNG xóa bỏ dòng có null?
Vì mỗi dòng là một đơn hàng thực tế. Xóa bỏ = mất thông tin kinh doanh quý giá. Thay vào đó, ta điền giá trị mặc định hợp lý.

In [20]:
# === 3a: Xóa trùng lặp ===
initial_count = merged.count()
merged = merged.dropDuplicates(["order_id"])
after_dedup = merged.count()
print(f"3a. Xóa trùng lặp: {initial_count - after_dedup:,} bản ghi đã loại bỏ")

# === 3b: Parse ngày tháng ===
print("3b. Parse cột ngày tháng...")
date_columns = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col_name in date_columns:
    if col_name in merged.columns:
        merged = merged.withColumn(col_name, F.to_timestamp(F.col(col_name), "yyyy-MM-dd HH:mm:ss"))
        print(f"   -> Đã parse: {col_name}")

# === 3c: Xử lý giá trị null ===
print("3c. Xử lý giá trị null...")
# Điền 0 cho cột số
numeric_fill = {
    "total_price": 0.0, "total_freight_value": 0.0, "total_payment_value": 0.0,
    "total_items": 0, "review_score": 0.0, "avg_product_weight_g": 0.0,
}
merged = merged.fillna(numeric_fill)
# Điền "unknown" cho cột text
string_fill = {
    "main_category_english": "unknown", "payment_type": "unknown",
    "seller_city": "unknown", "seller_state": "unknown",
}
merged = merged.fillna(string_fill)

# === 3d: Lọc bỏ đơn không hợp lệ ===
print("3d. Lọc bỏ đơn hàng không hợp lệ...")
merged = merged.filter(F.col("order_purchase_timestamp").isNotNull())
merged = merged.filter(
    (F.col("order_status") != "canceled") | (F.col("total_price") > 0)
)

final_count = merged.count()
print(f"\n Sau làm sạch: {final_count:,} dòng (đã loại {initial_count - final_count:,})")

3a. Xóa trùng lặp: 0 bản ghi đã loại bỏ
3b. Parse cột ngày tháng...
   -> Đã parse: order_purchase_timestamp
   -> Đã parse: order_approved_at
   -> Đã parse: order_delivered_carrier_date
   -> Đã parse: order_delivered_customer_date
   -> Đã parse: order_estimated_delivery_date
3c. Xử lý giá trị null...
3d. Lọc bỏ đơn hàng không hợp lệ...

 Sau làm sạch: 99,277 dòng (đã loại 164)


## 2.5 Feature Engineering

Đây là bước **quan trọng nhất** - tạo ra các đặc trưng mới từ dữ liệu thô để phục vụ phân tích và mô hình ML.

| Feature mới | Ý nghĩa | Tại sao quan trọng? |
|-------------|---------|---------------------|
| `delivery_days` | Số ngày giao hàng thực tế | Ảnh hưởng trực tiếp đến sự hài lòng |
| `estimated_vs_actual` | Chênh lệch dự kiến vs thực tế | Dương = sớm, âm = trễ |
| `freight_ratio` | Tỷ lệ phí ship / tổng giá trị | Phí ship cao → ít hấp dẫn |
| `order_value` | Tổng giá trị (giá + ship) | Chỉ số kinh doanh cơ bản |
| `purchase_hour/month/...` | Thông tin thời gian | Phát hiện xu hướng mua hàng |
| `delivery_status` | Đúng hạn / Trễ | Phân loại chất lượng dịch vụ |
| `satisfaction_level` | Mức độ hài lòng | Dựa trên review score |

In [21]:
# === 4a: Thời gian giao hàng ===
print("4a. Tính thời gian giao hàng...")
merged = merged.withColumn(
    "delivery_days",
    F.when(
        F.col("order_delivered_customer_date").isNotNull(),
        F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
    ).otherwise(F.lit(None))
)

# === 4b: Chênh lệch giao hàng ===
print("4b. Tính chênh lệch dự kiến vs thực tế...")
merged = merged.withColumn(
    "estimated_vs_actual",
    F.when(
        F.col("order_delivered_customer_date").isNotNull() & F.col("order_estimated_delivery_date").isNotNull(),
        F.datediff(F.col("order_estimated_delivery_date"), F.col("order_delivered_customer_date"))
    ).otherwise(F.lit(None))
)

# === 4c: Tỷ lệ phí vận chuyển ===
merged = merged.withColumn(
    "freight_ratio",
    F.when(F.col("total_price") > 0, F.round(F.col("total_freight_value") / F.col("total_price"), 4)).otherwise(0.0)
)

# === 4d: Tổng giá trị đơn hàng ===
merged = merged.withColumn("order_value", F.col("total_price") + F.col("total_freight_value"))

# === 4e: Features thời gian ===
print(" 4e. Tạo features thời gian...")
merged = (
    merged
    .withColumn("purchase_hour", F.hour("order_purchase_timestamp"))
    .withColumn("purchase_dayofweek", F.dayofweek("order_purchase_timestamp"))
    .withColumn("purchase_month", F.month("order_purchase_timestamp"))
    .withColumn("purchase_year", F.year("order_purchase_timestamp"))
    .withColumn("purchase_year_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
)

# === 4f: Phân loại giao hàng ===
merged = merged.withColumn(
    "delivery_status",
    F.when(F.col("estimated_vs_actual").isNull(), "not_delivered")
     .when(F.col("estimated_vs_actual") >= 0, "on_time")
     .otherwise("late")
)

# === 4g: Phân loại hài lòng ===
merged = merged.withColumn(
    "satisfaction_level",
    F.when(F.col("review_score") >= 4, "satisfied")
     .when(F.col("review_score") == 3, "neutral")
     .when(F.col("review_score") >= 1, "dissatisfied")
     .otherwise("no_review")
)

print(f"\n Tổng cột sau feature engineering: {len(merged.columns)}")
merged.select("order_id", "delivery_days", "estimated_vs_actual", "freight_ratio", 
              "order_value", "delivery_status", "satisfaction_level").show(5)

4a. Tính thời gian giao hàng...
4b. Tính chênh lệch dự kiến vs thực tế...
 4e. Tạo features thời gian...

 Tổng cột sau feature engineering: 42
+--------------------+-------------+-------------------+-------------+------------------+---------------+------------------+
|            order_id|delivery_days|estimated_vs_actual|freight_ratio|       order_value|delivery_status|satisfaction_level|
+--------------------+-------------+-------------------+-------------+------------------+---------------+------------------+
|000229ec398224ef6...|            8|                 14|       0.0898|            216.87|        on_time|         satisfied|
|00042b26cf59d7ce6...|           25|                 16|       0.0907|218.04000000000002|        on_time|         satisfied|
|0005f50442cb953dc...|            2|                 19|       0.2112|             65.39|        on_time|         satisfied|
|00061f2a7bc09da83...|            5|                 11|        0.148|             68.87|        on_time| 

## 2.6 Tính chỉ số RFM (Recency, Frequency, Monetary)

### RFM là gì?
RFM là phương pháp phân khúc khách hàng dựa trên **3 chỉ số hành vi**:
- **Recency (R)**: Khách hàng mua hàng gần đây nhất là khi nào? → Số ngày từ lần mua cuối
- **Frequency (F)**: Khách hàng mua hàng thường xuyên không? → Tổng số đơn hàng
- **Monetary (M)**: Khách hàng chi bao nhiêu tiền? → Tổng giá trị chi tiêu

### Tại sao dùng RFM?
RFM giúp doanh nghiệp:
- 🏆 Nhận diện khách VIP (R thấp, F cao, M cao)
- ⚠️ Phát hiện khách có nguy cơ rời bỏ (R cao)
- 🎯 Tối ưu chiến dịch marketing cho từng nhóm

### Cách tính:
1. Lọc chỉ các đơn hàng đã giao thành công (`order_status = "delivered"`)
2. Tính R, F, M cho từng `customer_unique_id`
3. Phân hạng bằng `ntile(4)` → mỗi chỉ số từ 1-4 điểm
4. Tổng hợp: `rfm_score = R + F + M` (3-12 điểm)

In [22]:
# Chỉ tính RFM cho đơn hàng đã giao thành công
delivered = merged.filter(F.col("order_status") == "delivered")

# Ngày mua hàng gần nhất (dùng làm mốc)
max_date = delivered.agg(F.max("order_purchase_timestamp")).collect()[0][0]
print(f" Ngày mốc: {max_date}")

# Tính RFM cho từng khách hàng
rfm_df = (
    delivered
    .groupBy("customer_unique_id")
    .agg(
        F.datediff(F.lit(max_date), F.max("order_purchase_timestamp")).alias("recency"),
        F.countDistinct("order_id").alias("frequency"),
        F.sum("total_payment_value").alias("monetary"),
        F.avg("review_score").alias("avg_review_score"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.first("customer_state").alias("customer_state"),
        F.first("customer_city").alias("customer_city"),
    )
)

# Phân hạng RFM (1-4 điểm cho mỗi chỉ số)
r_window = Window.orderBy(F.col("recency").asc())    # Recency thấp = tốt → điểm cao
f_window = Window.orderBy(F.col("frequency").desc())  # Frequency cao = tốt → điểm cao
m_window = Window.orderBy(F.col("monetary").desc())    # Monetary cao = tốt → điểm cao

rfm_df = (
    rfm_df
    .withColumn("r_score", F.ntile(4).over(r_window))
    .withColumn("f_score", F.ntile(4).over(f_window))
    .withColumn("m_score", F.ntile(4).over(m_window))
    .withColumn("rfm_score", F.col("r_score") + F.col("f_score") + F.col("m_score"))
)

print(f"\n Đã tính RFM cho {rfm_df.count():,} khách hàng")
rfm_df.select("recency", "frequency", "monetary", "rfm_score").describe().show()

# Join RFM vào merged_orders
merged = merged.join(
    rfm_df.select("customer_unique_id", "recency", "frequency", "monetary",
                  "r_score", "f_score", "m_score", "rfm_score"),
    on="customer_unique_id", how="left"
)

 Ngày mốc: 2018-08-29 22:00:37

 Đã tính RFM cho 93,358 khách hàng
+-------+------------------+-------------------+------------------+-----------------+
|summary|           recency|          frequency|          monetary|        rfm_score|
+-------+------------------+-------------------+------------------+-----------------+
|  count|             93358|              93358|             93358|            93358|
|   mean|237.47887701107564| 1.0334197390689603|165.19700261358162|7.499935731271021|
| stddev| 152.5950542957237|0.20909743968188677|226.31401249105576|2.505517331830654|
|    min|                 0|                  1|               0.0|                3|
|    max|               713|                 15|          13664.08|               12|
+-------+------------------+-------------------+------------------+-----------------+



## 2.7 Tạo nhãn Churn (Rời bỏ)

### Định nghĩa Churn
- **Churn = 1**: Khách hàng KHÔNG mua hàng trong **90 ngày cuối** → Đã rời bỏ
- **Churn = 0**: Khách hàng vẫn hoạt động trong 90 ngày gần nhất

### Tại sao chọn ngưỡng 90 ngày?
Trong thương mại điện tử, chu kỳ mua hàng trung bình là 30-60 ngày. Ngưỡng 90 ngày (3 tháng) đủ lớn để xác nhận rằng khách hàng thực sự đã ngừng mua, không chỉ là "đang chờ".

In [23]:
# Tạo nhãn churn dựa trên recency
merged = merged.withColumn("churn", F.when(F.col("recency") > 90, 1).otherwise(0))

# Thống kê phân bố churn
print(" Phân bố Churn:")
merged.groupBy("churn").count().show()

total = merged.count()
churn_count = merged.filter(F.col("churn") == 1).count()
churn_rate = churn_count / total * 100 if total > 0 else 0

 Phân bố Churn:
+-----+-----+
|churn|count|
+-----+-----+
|    1|77369|
|    0|21908|
+-----+-----+



## 2.8 Lưu dữ liệu lên HDFS (Silver & Gold)

Dữ liệu đã xử lý được lưu dưới dạng **Parquet** lên HDFS:
- `merged_orders` → **Silver Layer** (dữ liệu đã clean + features)
- `rfm_customers` → **Gold Layer** (dữ liệu tổng hợp sẵn sàng cho ML)

> **Tại sao dùng `.coalesce(4)`?** Để giảm số file part nhỏ trên HDFS, tránh tình trạng "small files problem" ảnh hưởng hiệu năng NameNode.

In [24]:
HDFS_SILVER = "hdfs://localhost:9000/user/bigdata/olist/silver"
HDFS_GOLD = "hdfs://localhost:9000/user/bigdata/olist/gold"

# Lưu merged_orders -> Silver
merged_path = f"{HDFS_SILVER}/merged_orders"
print(f"Đang lưu merged_orders -> {merged_path}...")
merged.coalesce(4).write.mode("overwrite").parquet(merged_path)
print(f"   Đã lưu merged_orders")

# Lưu rfm_customers -> Gold
rfm_path = f"{HDFS_GOLD}/rfm_customers"
print(f"Đang lưu rfm_customers -> {rfm_path}...")
rfm_df.coalesce(2).write.mode("overwrite").parquet(rfm_path)
print(f"   Đã lưu rfm_customers")

# Xác nhận
check = spark.read.parquet(merged_path)
print(f"\n Xác nhận: merged_orders có {check.count():,} dòng trên HDFS")

Đang lưu merged_orders -> hdfs://localhost:9000/user/bigdata/olist/silver/merged_orders...
   Đã lưu merged_orders
Đang lưu rfm_customers -> hdfs://localhost:9000/user/bigdata/olist/gold/rfm_customers...
   Đã lưu rfm_customers

 Xác nhận: merged_orders có 99,277 dòng trên HDFS


## 2.9 Tổng kết ETL Pipeline

| Chỉ số | Giá trị |
|--------|---------|
| Tổng đơn hàng sau ETL | ~99,000 |
| Tổng khách hàng (RFM) | ~93,000 |
| Số features mới tạo | 15+ |
| Tỷ lệ Churn | ~60% |
| Thời gian chạy | ~3-5 phút |

### Kiến trúc dữ liệu sau ETL:
```
HDFS
├── Bronze (CSV thô)        ← Bước 1: Ingestion
├── Silver (Parquet đã ETL)  ← Bước 2: ETL (notebook này)
│   └── merged_orders/
└── Gold (Parquet tổng hợp)
    └── rfm_customers/
```

**Bước tiếp theo** → Notebook `03_eda.ipynb`: Phân tích khám phá dữ liệu.

In [25]:
# Dừng SparkSession
spark.stop()
print("SparkSession đã dừng.")

SparkSession đã dừng.
